# RAG Consumer Demo — built on `ragmodel` / `ragtorch`

A real, end-to-end RAG (retrieval-augmented generation) application that
uses the **published** `ragmodel` package (`pip install ragmodel`,
`import ragtorch`) as its execution/orchestration layer.

Every component below — scraping, chunking, embeddings, the vector
store, reranking, prompt construction, and generation — is
**application-level code**, not part of `ragtorch` itself. This
notebook is a real consumer test of the published library, run from a
completely fresh Colab runtime.

**Runtime → Run all** should work top to bottom with zero manual setup
beyond optionally adding an `OPENAI_API_KEY` Colab Secret if you want
`MODE="llm"` instead of the default offline mode.

## 1. Install

Installs the published package plus this demo's application dependencies.

In [ ]:
!pip install -q ragmodel==0.5.0 requests beautifulsoup4 sentence-transformers faiss-cpu openai

## 2. Verify installation

Confirms the package actually installed from PyPI (not a local checkout), the
version is correct, and the import path resolves to `site-packages`.

In [ ]:
import ragtorch

print("ragtorch version:", ragtorch.__version__)
print("ragtorch file:", ragtorch.__file__)
assert ragtorch.__version__ == "0.5.0", f"unexpected version: {ragtorch.__version__}"
assert "site-packages" in ragtorch.__file__.replace("\\", "/"), "not installed from PyPI"
print("\nVerified: ragtorch 0.5.0 installed from the published PyPI package (ragmodel).")

## 3. Configuration

Fetch this demo's application source (not ragtorch itself -- these are
plain application files, this notebook's own code, pulled from the same
repository ragtorch is published from, purely for convenience so this
Colab cell doesn't need to redefine every class inline).

In [ ]:
import os
import sys
import urllib.request

REPO_RAW = (
    "https://raw.githubusercontent.com/payamfirouzfar/RAG-MODULE/main/examples/rag_consumer/src"
)
os.makedirs("src", exist_ok=True)
open("src/__init__.py", "w").close()

for module in [
    "config.py",
    "scraper.py",
    "dataset.py",
    "chunking.py",
    "embeddings.py",
    "vector_store.py",
    "retriever.py",
    "reranker.py",
    "prompt_builder.py",
    "generator.py",
    "pipeline.py",
    "evaluation.py",
]:
    urllib.request.urlretrieve(f"{REPO_RAW}/{module}", f"src/{module}")

sys.path.insert(0, ".")
print("Application source fetched.")

In [ ]:
from src.config import Config

CONFIG = Config(
    urls=[
        "https://docs.python.org/3/tutorial/introduction.html",
        "https://docs.python.org/3/tutorial/controlflow.html",
        "https://docs.python.org/3/tutorial/datastructures.html",
        "https://docs.python.org/3/tutorial/modules.html",
        "https://docs.python.org/3/tutorial/errors.html",
    ],
    chunk_size=800,
    chunk_overlap=120,
    embedding_model="sentence-transformers/all-MiniLM-L6-v2",
    vector_store_backend="faiss",
    top_k=5,
    min_score=0.2,  # results below this cosine-similarity score are dropped -- see Section 15
    rerank=False,
    mode="offline",  # change to "llm" if you have an OPENAI_API_KEY Colab Secret
)
print(CONFIG)

## 4. Scrape dataset

SCRAPE MODE: fetches the configured URLs (bounded, respects robots.txt,
descriptive User-Agent, timeouts, inter-request delay), cleans the HTML,
and saves the result to `data/documents.jsonl`.

If you already have a saved dataset and don't want to touch the network
again, skip this cell and run the OFFLINE DATASET MODE cell below instead.

In [ ]:
from pathlib import Path

from src.dataset import build_dataset, save_dataset

documents = build_dataset(
    CONFIG.urls,
    user_agent=CONFIG.scrape_user_agent,
    timeout=CONFIG.scrape_timeout_seconds,
    delay_seconds=CONFIG.scrape_delay_seconds,
    cache_dir=Path("data/raw_html"),
    max_pages=CONFIG.max_pages,
)
save_dataset(documents, Path(CONFIG.dataset_path))
print(f"Scraped and saved {len(documents)} documents to {CONFIG.dataset_path}")

**OFFLINE DATASET MODE** (alternative to the cell above): loads an
already-saved dataset with zero network access.

In [ ]:
from pathlib import Path


# Uncomment to use an already-saved dataset instead of re-scraping:
# documents = load_dataset(Path(CONFIG.dataset_path))
# print(f"Loaded {len(documents)} documents from {CONFIG.dataset_path} (no network used)")

## 5. Inspect dataset

In [ ]:
for doc in documents:
    print(f"- {doc.title!r} ({doc.url})")
    print(f"  {len(doc.text)} characters, retrieved_at={doc.retrieved_at}")
print(f"\nTotal: {len(documents)} documents")

## 6. Chunk documents

In [ ]:
from src.chunking import chunk_documents

chunks = chunk_documents(
    documents, chunk_size=CONFIG.chunk_size, chunk_overlap=CONFIG.chunk_overlap
)
print(
    f"{len(documents)} documents -> {len(chunks)} chunks "
    f"(chunk_size={CONFIG.chunk_size}, chunk_overlap={CONFIG.chunk_overlap})"
)
print("\nExample chunk:")
print(chunks[0])

## 7. Load embedding model

Real embedding model via sentence-transformers (downloads a small model
on first use).

In [ ]:
from src.embeddings import SentenceTransformersEmbedder

embedder = SentenceTransformersEmbedder(CONFIG.embedding_model)
print(f"Loaded embedder: {embedder.model_name}")

sample_vector = embedder.embed_query("test query")
print(f"Embedding dimensionality: {len(sample_vector)}")

## 8. Build vector store

In [ ]:
from src.vector_store import build_vector_store

vectors = embedder.embed_documents([c.text for c in chunks])
vector_store = build_vector_store(CONFIG.vector_store_backend, dimensions=len(vectors[0]))
vector_store.add(chunks, vectors)
print(f"Indexed {len(chunks)} chunks into a {CONFIG.vector_store_backend} vector store.")

## 9. Test retrieval

Sanity-check retrieval directly, before wiring it into the full ragtorch pipeline.

In [ ]:
from src.retriever import Retriever

retriever = Retriever(embedder, vector_store, top_k=CONFIG.top_k, min_score=CONFIG.min_score)
results = retriever("How do I define a function in Python?")
for r in results:
    print(f"[{r.score:.3f}] {r.title} ({r.chunk_id})")
    print(f"    {r.text[:150]}...")

## 10. Build RAG components

Assembles the reranker (optional) and generator (offline or LLM, per
`CONFIG.mode`).

In [ ]:
from src.generator import build_generator
from src.reranker import Reranker

reranker = Reranker() if CONFIG.rerank else None
generator = build_generator(CONFIG)
print(f"reranker: {reranker}")
print(f"generator: {type(generator).__name__} (mode={CONFIG.mode})")

## 11. Build ragtorch pipeline

Composes Retriever -> Reranker -> PromptBuilder -> Generator using
`ragtorch.Sequential`. See the demo README's "Design notes" section for
why Sequential (verified against Sequential.forward's real data-flow
contract) fits this pipeline cleanly via a threaded `PipelineState`
value object, rather than forcing Block/CompositionGraph.

In [ ]:
from src.pipeline import build_pipeline

pipeline = build_pipeline(
    retriever=retriever, reranker=reranker, generator=generator, mode=CONFIG.mode
)
print(f"Pipeline built: {pipeline}")
print(pipeline.inspect())

## 12. Run one question

Runs the pipeline through `ragtorch.ExecutionEngine`, capturing Run/Trace/Metrics.

In [ ]:
from src.pipeline import run_pipeline

result = run_pipeline(pipeline, "How do I define a function in Python?")
print("Question:", result.question)
print("Answer:  ", result.answer)
print("Run status:", result.run_status)

## 13. Show sources

Every cited source maps to something actually retrieved -- nothing is invented.

In [ ]:
print("Sources:")
for s in result.sources:
    print(f"  [{s['index']}] {s['title']} — {s['url']}")

## 14. Run multiple questions

In [ ]:
questions = [
    "What is a Python list comprehension?",
    "How does Python handle exceptions?",
    "What is the difference between a tuple and a list?",
]

for q in questions:
    r = run_pipeline(pipeline, q)
    print(f"Q: {q}")
    print(f"A: {r.answer}")
    print(f"   status={r.run_status}, sources={len(r.sources)}")
    print()

## 15. Negative question

A question whose answer is NOT present in this dataset. The system
should say the available evidence is insufficient -- not guess.

Note: `Retriever`'s `min_score` (Section 3, `CONFIG.min_score`) drops
results below a similarity threshold. This matters because a plain
nearest-neighbor vector search always returns its top_k *closest*
matches even when none are actually relevant -- filtering by score is
what turns "the least-bad match" into "no relevant results," which is
what lets `OfflineGenerator` correctly say evidence is insufficient
instead of extracting an answer from an irrelevant chunk.

In [ ]:
from src.generator import INSUFFICIENT_EVIDENCE_MESSAGE

negative_question = "What is the boiling point of mercury in Fahrenheit?"
r = run_pipeline(pipeline, negative_question)
print(f"Q: {negative_question}")
print(f"A: {r.answer}")
print()
if CONFIG.mode == "offline":
    assert INSUFFICIENT_EVIDENCE_MESSAGE.lower() in r.answer.lower() or len(r.sources) == 0, (
        "expected an insufficient-evidence signal for a question with no relevant documents"
    )
    print(
        "Confirmed: system correctly signaled insufficient evidence "
        "rather than fabricating an answer."
    )
else:
    print(
        "MODE='llm': check the answer above manually -- the prompt instructs the model to "
        "say when evidence is insufficient, but this is not deterministically enforceable "
        "the way OfflineGenerator's behavior is."
    )

## 16. Evaluation

A small, explicitly-labeled application-level smoke/evaluation dataset
-- NOT a scientific RAG benchmark. Includes negative (unanswerable)
questions.

In [ ]:
from src.evaluation import EvalCase, evaluate_case, summarize
from src.generator import INSUFFICIENT_EVIDENCE_MESSAGE


def _case(question, doc_index):
    return EvalCase(question=question, expected_document_id=documents[doc_index].document_id)


def _negative_case(question):
    return EvalCase(question=question, expected_document_id=None, is_negative=True)


eval_cases = [
    _case("How do I define a function in Python?", 1),
    _case("What is a for loop used for?", 1),
    _case("What is a Python list?", 2),
    _case("How do dictionaries work in Python?", 2),
    _case("How do I import a module?", 3),
    _case("What is a package in Python?", 3),
    _case("How does Python handle exceptions?", 4),
    _case("What is a try/except block?", 4),
    _case("What does the print function do?", 0),
    _case("What is an interactive interpreter?", 0),
    # Negative (unanswerable) questions -- the dataset does not cover these topics.
    _negative_case("What is the boiling point of mercury?"),
    _negative_case("Who won the World Cup in 1990?"),
]

eval_results = []
for case in eval_cases:
    retrieval_results = retriever(case.question)
    pipeline_result = run_pipeline(pipeline, case.question)
    eval_results.append(
        evaluate_case(
            case,
            retrieval_results,
            pipeline_result,
            insufficient_evidence_marker=INSUFFICIENT_EVIDENCE_MESSAGE,
        )
    )

summary = summarize(eval_results)
print(summary.render())

## 17. Benchmark

Measures chunking/embedding/indexing/retrieval/pipeline latency. No hard threshold is asserted.

In [ ]:
import statistics
import time


def time_it(fn, iterations=10):
    samples = []
    for _ in range(iterations):
        start = time.perf_counter()
        fn()
        samples.append((time.perf_counter() - start) * 1000)
    return statistics.mean(samples), statistics.median(samples)


mean, median = time_it(lambda: retriever("sample benchmark query"), iterations=20)
print(f"retrieval latency:          mean={mean:.2f}ms  median={median:.2f}ms")

mean, median = time_it(lambda: run_pipeline(pipeline, "sample benchmark query"), iterations=10)
print(f"complete pipeline latency:  mean={mean:.2f}ms  median={median:.2f}ms  (mode={CONFIG.mode})")

## 18. Execution trace

Inspects the real `ragtorch.Trace`/`Run`/`Metrics` captured by
`ExecutionEngine` for one pipeline run.

In [ ]:
from src.pipeline import PipelineState

from ragtorch import ExecutionEngine, ObservabilityLevel

engine = ExecutionEngine(level=ObservabilityLevel.DEBUG)
execution_result = engine.execute(pipeline, PipelineState(question="What is a Python list?"))

print("Run status:", execution_result.run.status)
print("Run duration:", execution_result.run.duration(), "seconds")
print()
print("Trace:")
print(execution_result.trace.render())
print()
print("Metrics summary:")
print(execution_result.metrics.summarize_all())

## 19. Troubleshooting

- **`ModuleNotFoundError: No module named 'ragtorch'`**: re-run the
  install cell (Section 1) -- Colab runtimes reset between sessions.
- **`ragtorch.__version__` is not `0.5.0`**: Colab may have cached an
  older wheel. Restart the runtime (Runtime -> Restart session) and
  re-run from Section 1.
- **Scraping fails with a network error**: switch to OFFLINE DATASET
  MODE (Section 4) if you have a previously-saved
  `data/documents.jsonl`, or check your network connectivity.
- **`GeneratorError: OPENAI_API_KEY is not set`**: either add an
  `OPENAI_API_KEY` Colab Secret (key icon in the left sidebar) and
  grant this notebook access to it, or set `CONFIG.mode = "offline"`
  in Section 3 and re-run from there.
- **`faiss` import errors on an unusual runtime**: set
  `CONFIG.vector_store_backend = "in_memory"` in Section 3 and re-run
  from there -- `InMemoryVectorStore` has no external dependency.
- **Retrieval results look poor**: this demo's default dataset (5
  Python documentation pages) is intentionally small for a fast Colab
  demo -- see the README's "Limitations" section. Try a more specific
  question, or add more URLs to `CONFIG.urls` in Section 3 and
  re-scrape (Section 4).